In [2]:
!pip install langchain langchain-ollama --quiet

In [1]:
with open("sample.txt", "w") as file:
    file.write("Hello, this is a sample text file.\n")
    file.write("This file was created using Python.")

In [2]:
from langchain.tools import tool
import os

@tool
def read_file(file_path: str) -> str:
    """Read the contents of a file."""
    with open(file_path, "r") as f:
        content = f.read()

    return {content}

@tool
def get_content_length(input_data: str) -> int:
    """
    Returns the number of characters.
    If input_data is a .txt file path, it reads the file and counts the characters.
    Otherwise it treats input_data as plain text.
    """
    
    # Check if it's a txt file path
    if input_data.endswith(".txt") and os.path.isfile(input_data):
        with open(input_data, "r") as f:
            content = f.read()
        return len(content)

    # Otherwise treat as plain text
    return len(input_data)
   
print('Tool name   :', read_file.name)
print('Description :', read_file.description)
print('Schema      :', read_file.args)

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Tool name   : read_file
Description : Read the contents of a file.
Schema      : {'file_path': {'title': 'File Path', 'type': 'string'}}


In [3]:
result = read_file.invoke({'file_path':'sample.txt'})
print(result)

Content_length = get_content_length.invoke({'input_data':'Manaswi'})
print(Content_length)

{'Hello, this is a sample text file.\nThis file was created using Python.'}
7


In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model='llama3.2')
tools = [read_file, get_content_length]

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke('read the content in sample.txt')
print('Tool calls :', response.tool_calls)

Tool calls : [{'name': 'read_file', 'args': {'file_path': 'sample.txt'}, 'id': 'ebeb6158-c31d-4aff-8260-36ee819a9294', 'type': 'tool_call'}]


In [5]:
from langchain_core.messages import HumanMessage, ToolMessage

tool_map = {'read_file': read_file, 'get_content_length': get_content_length}

# Step 1 — ask the model
messages = [HumanMessage(content="introduce yourself in telugu")]
response = llm_with_tools.invoke(messages)

messages.append(response)

# Step 2 — run the tool the model asked for
for tool_call in response.tool_calls:
    tool_fn = tool_map[tool_call['name']]
    tool_result = tool_fn.invoke(tool_call['args'])
    messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_call['id']))

print(messages)
# Step 3 — send result back to the model for a final answer
final = llm_with_tools.invoke(messages)
print('Final answer:', final.content)

[HumanMessage(content='introduce yourself in telugu', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-03-12T23:54:03.6510214Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3507948200, 'load_duration': 123873700, 'prompt_eval_count': 216, 'prompt_eval_duration': 320043800, 'eval_count': 171, 'eval_duration': 2999700000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019ce478-ae4d-74c2-a514-eb31945b5118-0', tool_calls=[{'name': 'get_content_length', 'args': {'input_data': 'నా పేరు శ్రీనివాస్, నేను ఒక ఆయాజియత్వ ఉపకరణం, వారి విషయాలను చెప్పడానికి ఉపయోగించే ఉపకరణం'}, 'id': '53981547-e3f3-478e-b25c-fd52ba02c2ad', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 216, 'output_tokens': 171, 'total_tokens': 387}), ToolMessage(content='88', tool_call_id='53981547-e3f3-478e-b25c-fd52ba02c2ad')]
Final answer: నా పేరు శ్రీని

In [69]:
s="Hello, this is a sample text file\n."
p="This file was created using Python."
i= len(s) +len(p)
print(i)

70


In [61]:
questions = [
    'What are the contents in sample.txt?',
    'What is the length of the word Manaswi?'
]

for question in questions:
    response = llm_with_tools.invoke(question)
    if response.tool_calls:
        call = response.tool_calls[0]
        print(f'Q: {question}')
        print(f'   -> Model chose tool: {call["name"]} with args {call["args"]}')
    else:
        print(f'Q: {question}')
        print(f'   -> Direct answer: {response.content}')
    print()

Q: What are the contents in sample.txt?
   -> Model chose tool: read_file with args {'file_path': 'sample.txt'}

Q: What is the length of the word Manaswi?
   -> Model chose tool: get_content_length with args {'input_data': 'Manaswi'}

